# watchin-u: 1. Prepare the dataset (Colab)

Turns raw camera footage in Drive into frames, a pre-filled visit log, contact sheets and
**pre-labelled CVAT bundles**. After labelling it builds the training datasets.

**Part A** (now): footage -> frames -> visit log + contact sheets -> CVAT pre-labels
**Part B** (after labelling in CVAT and filling the visit log): datasets for training

Runtime: a GPU is only needed for pre-labelling (Runtime -> Change runtime type -> T4 GPU).
Footage of the plant stays in your own Drive and Colab session.

## Setup: Drive, code, install

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# ---- settings ------------------------------------------------------------------
DRIVE = "/content/drive/MyDrive/watchinu"  # project folder in your Drive (created if missing)
REPO = "nav0225/watchin-u"
BRANCH = "ccr-09c28bb1-bjcddo"
# Private repo? Add a Colab secret named GITHUB_TOKEN (key icon in the left bar, read-only
# token). Or upload the code as  DRIVE/code/watchin-u.zip  and it is used instead.

import os, pathlib, shutil, subprocess, zipfile

pathlib.Path(DRIVE).mkdir(parents=True, exist_ok=True)
CODE = pathlib.Path("/content/watchin-u")
if CODE.exists():
    shutil.rmtree(CODE)
code_zip = pathlib.Path(DRIVE) / "code" / "watchin-u.zip"
if code_zip.exists():
    with zipfile.ZipFile(code_zip) as zf:
        zf.extractall("/content/_code")
    root = next(p.parent for p in pathlib.Path("/content/_code").rglob("pyproject.toml"))
    shutil.move(str(root), CODE)
    print("code from", code_zip)
else:
    token = ""
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        pass
    url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(CODE)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        msg = r.stderr.replace(token, "***") if token else r.stderr
        raise RuntimeError("git clone failed - private repo? add GITHUB_TOKEN or the code zip\n" + msg)
print(subprocess.run(["git", "-C", str(CODE), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -e "/content/watchin-u[ml]"
%cd /content/watchin-u
!watchinu --help | head -5

## Part A1: unpack footage and extract frames

In [ ]:
FOOTAGE_ZIP = "/content/drive/MyDrive/2026-11-04-dataset.zip"  # the zip you uploaded
EVERY_S = 1.0       # 10-s clips: 1 frame/s. Long recordings / time-lapse: 5-10
CAMERA_ID = "cam01"
BAY = "left"
SESSION = "2026-11-04"  # one folder per recording session in Drive

LOCAL = pathlib.Path("/content/work") / SESSION
FOOTAGE = LOCAL / "footage"
FRAMES = LOCAL / "frames"
FOOTAGE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(FOOTAGE_ZIP) as zf:
    zf.extractall(FOOTAGE)  # zip keeps file times: used as clip start when names have none
!find {FOOTAGE} -type f | head -50

In [ ]:
# What is in the footage? (resolution, codec, length) - camera exports should be 2560x1440
import json as _json

for f in sorted(p for p in FOOTAGE.rglob("*") if p.suffix.lower() in {".mp4", ".mkv", ".avi", ".mov", ".ts"}):
    r = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                        "stream=width,height,codec_name,avg_frame_rate:format=duration",
                        "-of", "json", str(f)], capture_output=True, text=True)
    info = _json.loads(r.stdout or "{}")
    st, fm = (info.get("streams") or [{}])[0], info.get("format", {})
    print(f"{f.name:40s} {st.get('width')}x{st.get('height')} {st.get('codec_name')} "
          f"{st.get('avg_frame_rate')} fps  {float(fm.get('duration', 0)):.1f}s")

In [ ]:
!watchinu extract --input {FOOTAGE} --out {FRAMES} --every-s {EVERY_S} --camera-id {CAMERA_ID} --start-time-from-mtime
!watchinu dedupe --manifest {FRAMES}/manifest.jsonl

## Part A2: visit log + contact sheets

`visit_log.csv` gets one row per clip with the times filled in. Open it in Google Sheets or Excel
and fill **product** (`wire_rod`, `ingot`, `empty`, ...), **layout** (`linear` / `criss_cross` /
`mixed`) and **coil_count** (from the loading slip). If a clip shows the tarp going on, fill
`tarp_start` / `tarp_done` too.

The contact sheets show 4 frames per clip. **Paste them into the Claude chat** so the
geometry and templates can be checked without downloading the footage.

In [ ]:
OUT = pathlib.Path(DRIVE) / "sessions" / SESSION
OUT.mkdir(parents=True, exist_ok=True)
!watchinu visit-log-from-clips --manifest {FRAMES}/manifest.jsonl --bay {BAY} --out {OUT}/visit_log.csv
!watchinu contact-sheet --manifest {FRAMES}/manifest.jsonl --out {OUT}/contact_sheets
from IPython.display import Image, display

for p in sorted((OUT / "contact_sheets").glob("*.jpg"))[:2]:
    display(Image(str(p), width=1100))

## Part A3: calibration check

Draws the configured bay polygon, bed corners and slot centres on a real frame. Send the
image to Claude. It is used to re-fit `configs/camera01.yaml` (the current geometry is a first
estimate from a phone photo).

In [ ]:
import json as _json
from IPython.display import Image, display

recs = [_json.loads(l) for l in open(FRAMES / "manifest.jsonl")]
big = [r for r in recs if r["width"] >= 2560] or recs
sample = FRAMES / big[len(big) // 2]["image"]
for layout in ("linear", "criss_cross"):
    !watchinu render-config --config configs/camera01.yaml --image {sample} --out {OUT}/calib_{layout}.png --layout {layout}
display(Image(str(OUT / "calib_linear.png"), width=1100))

## Part A4: pre-label for CVAT

A zero-shot open-vocabulary detector proposes `coil` / `person` boxes, so labelling becomes
*correcting* rather than drawing. In CVAT: create a task with labels `coil` and `person`,
upload `images_*.zip`, then **Actions -> Upload annotations -> YOLO 1.1** with `prelabel_*.zip`.

Once about 200 frames are corrected and a v0 detector is trained (notebook 2), re-run this cell
with `WEIGHTS` set to that `best.pt` (and `PROMPTS = None`). The pre-labels get much better.

In [ ]:
WEIGHTS = "yolov8s-worldv2.pt"   # later: f"{DRIVE}/runs/detect-v1/weights/best.pt"
PROMPTS = "wrapped wire coil=coil,person=person"  # set to None for our own trained weights
CVAT = OUT / "cvat"
CVAT.mkdir(exist_ok=True)
prompt_arg = f'--prompts "{PROMPTS}"' if PROMPTS else ""
!watchinu prelabel --weights {WEIGHTS} {prompt_arg} --manifest {FRAMES}/manifest.jsonl --out {CVAT}/prelabel_v0.zip --images-zip {CVAT}/images_v0.zip --device 0 --min-conf 0.15
!ls -la {CVAT}

In [ ]:
# Keep the frames in Drive (needed again in Part B and for later sessions)
shutil.copytree(FRAMES, OUT / "frames", dirs_exist_ok=True)
print("saved to", OUT)

## Part B: build training datasets (after labelling)

Needs, in `DRIVE/sessions/<SESSION>/`:
* `frames/` (from Part A),
* `visit_log.csv` **filled in**,
* `labels/`: unzip the CVAT export (**YOLO 1.1**) here; the `.txt` files are in `obj_train_data/`.

Multiple sessions: run Part A for each, then list them all in `SESSIONS` below.

In [ ]:
import json as _json

SESSIONS = ["2026-11-04"]      # every labelled session, e.g. ["2026-11-04", "2026-11-06"]
DATA = pathlib.Path(DRIVE) / "datasets"
merged = pathlib.Path("/content/merged")
if merged.exists():
    shutil.rmtree(merged)
(merged / "labels").mkdir(parents=True)
manifest_lines, logs = [], []
for sess in SESSIONS:
    src = pathlib.Path(DRIVE) / "sessions" / sess
    shutil.copytree(src / "frames" / "images", merged / "images", dirs_exist_ok=True)
    manifest_lines += open(src / "frames" / "manifest.jsonl").read().splitlines()
    lbl = src / "labels" / "obj_train_data"
    if lbl.exists():
        for f in lbl.glob("*.txt"):
            shutil.copy(f, merged / "labels" / f.name)
    logs.append(open(src / "visit_log.csv").read().splitlines())
(merged / "manifest.jsonl").write_text("\n".join(manifest_lines) + "\n")
header = logs[0][0]
(merged / "visit_log.csv").write_text("\n".join([header] + [l for lg in logs for l in lg[1:]]) + "\n")
print(len(manifest_lines), "frames,", sum(len(l) - 1 for l in logs), "visits")

In [ ]:
M = merged / "manifest.jsonl"
V = merged / "visit_log.csv"
FROZEN = pathlib.Path(DRIVE) / "gold_test_visits.txt"   # created on the first run, then kept
frozen_arg = f"--frozen-test {FROZEN}" if FROZEN.exists() else ""
!watchinu split --manifest {M} --visit-log {V} --val 0.15 --test 0.2 {frozen_arg}
if not FROZEN.exists():
    rows = [_json.loads(line) for line in open(M)]
    test_visits = sorted({r["visit_id"] for r in rows if r["split"] == "test" and r["visit_id"]})
    FROZEN.write_text("\n".join(test_visits) + "\n")
    print("froze", len(test_visits), "gold test visits ->", FROZEN)

!watchinu export-detect --manifest {M} --labels {merged}/labels --out /content/ds/detect --classes coil,person --config configs/camera01.yaml --bay left
!watchinu export-cls --manifest {M} --visit-log {V} --task stage --out /content/ds/stage --config configs/camera01.yaml --bay left
!watchinu export-cls --manifest {M} --visit-log {V} --task layout --out /content/ds/layout --config configs/camera01.yaml --bay left
DATA.mkdir(parents=True, exist_ok=True)
for name in ("detect", "stage", "layout"):
    shutil.make_archive(str(DATA / name), "zip", f"/content/ds/{name}")
!ls -la {DATA}